# 4.5 — Spatial Distribution of Forecast Errors

Per-station MAE maps at +30 min, +2 h, +6 h for the main models
at MR=0, with DEM elevation background.

In [ ]:
import os, sys, datetime as dt
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import matplotlib.colors as mcolors
for _cand in (os.getcwd(),
              os.path.join(os.getcwd(), "notebooks", "analysis"),
              os.path.dirname(os.path.abspath("__file__"))):
    if os.path.isfile(os.path.join(_cand, "common.py")):
        if _cand not in sys.path: sys.path.insert(0, _cand)
        break
import importlib, common as C; importlib.reload(C)
plt.style.use("default")
plt.rcParams.update({"figure.dpi": 110, "font.size": 10,
                     "figure.facecolor": "white", "axes.facecolor": "white",
                     "savefig.facecolor": "white", "axes.edgecolor": "0.3",
                     "axes.labelcolor": "black", "xtick.color": "0.3",
                     "ytick.color": "0.3", "text.color": "black"})

RUNS  = C.discovered_runs()
MR0_RUNS = [r for r in RUNS if "mr0.00" in RUNS[r]]
ns = C.norm_stats(); VARS = ns["var_names"]; STD = ns["std"]
stn = C.station_table(); KEEP = C.keep_mask(stn, VARS)
AGG  = {r: C.load_agg(r, "mr0.00") for r in MR0_RUNS}
GRID = AGG[MR0_RUNS[0]]["grid"]; LEAD = C.lead_labels(GRID); K = len(GRID)
NV = len(VARS)
print("Models at MR=0.00:", MR0_RUNS)

In [ ]:
import geopandas as gpd
import rioxarray  # noqa

PROJ = os.path.abspath(os.path.join(os.getcwd(), "..", "..")) \
       if os.path.isfile("common.py") else os.getcwd()
if os.path.join(PROJ, "src") not in sys.path:
    sys.path.insert(0, os.path.join(PROJ, "src"))

PATH_SWISSSHAPE = os.path.expanduser(
    os.environ.get("SWISSSHAPE",
        os.path.join(PROJ, "swissboundaries3d_2056.shp.zip")))
_CAND = [os.environ.get("DATA_ROOT", ""),
         os.path.expanduser("~/PeakWeatherDataset"),
         os.path.join(PROJ, "PeakWeatherDataset")]
DATA_ROOT = next((p for p in _CAND if os.path.isdir(str(p))), _CAND[-1])

from peakweather.dataset import PeakWeatherDataset
ds_topo = PeakWeatherDataset(
    root=DATA_ROOT,
    parameters=["temperature", "pressure", "humidity",
                 "wind_speed", "wind_direction", "precipitation"],
    compute_uv=True, station_type="meteo_station",
    imputation_method=None, freq="d", extended_topo_vars="DEM")

def _load_dem_and_border(ds_topo, path_swissshape, coarsen=10):
    switzerland = gpd.read_file(
        path_swissshape,
        layer='swissBOUNDARIES3D_1_5_TLM_LANDESGEBIET').to_crs('EPSG:2056')
    minx, miny, maxx, maxy = switzerland.total_bounds
    topo = ds_topo.load_topography()
    dem  = topo['topo_DEM'].dem
    dem_ch = dem.rio.clip(switzerland.geometry, switzerland.crs, drop=False)
    dem_bg = dem.coarsen(x=coarsen, y=coarsen, boundary='trim').mean()
    dem_fg = dem_ch.coarsen(x=coarsen, y=coarsen, boundary='trim').mean()
    dem_bg = dem_bg.sel(x=slice(minx, maxx), y=slice(miny, maxy))
    dem_fg = dem_fg.sel(x=slice(minx, maxx), y=slice(miny, maxy))
    return dem_bg, dem_fg, switzerland

def draw_dem(ax, dem_bg, dem_fg, switzerland):
    norm = mcolors.Normalize(vmin=0, vmax=4500)
    dem_bg.plot(ax=ax, cmap='terrain', norm=norm, alpha=0.35,
                robust=True, add_labels=False, add_colorbar=False)
    dem_fg.plot(ax=ax, cmap='terrain', norm=norm,
                robust=True, add_labels=False, add_colorbar=False)
    switzerland.boundary.plot(ax=ax, color='white', linewidth=1.0)
    ax.axis('off')

print('Loading DEM + border ...')
dem_bg, dem_fg, switzerland = _load_dem_and_border(ds_topo, PATH_SWISSSHAPE)
print('Done.')

## Exclusion overlay

Every per-station scatter map below also marks stations excluded for that
variable (`common.excluded_station_variable_reasons()`) with an X:
**black** = missing in both train and test, **white** = missing in train
only, **grey** = missing in test only, **orange** = present in both splits
but excluded for a data-quality reason (BIZ/pressure, sensor drift).

In [ ]:
REASON_STYLE = {
    "missing_train": dict(color="white",   label="missing in train only"),
    "missing_test":  dict(color="0.6",     label="missing in test only"),
    "missing_both":  dict(color="black",   label="missing in train & test"),
    "drift":         dict(color="#E8A838", label="excluded \u2014 sensor drift"),
}
EXCL_REASONS = C.excluded_station_variable_reasons()

def overlay_exclusion_markers(ax, variable, s=70, legend=False):
    return  # X markers removed per user request

## Per-station MAE maps — LSTM, Spatially Blind, Dense

Rows = models (LSTM → Spatially Blind → Dense), columns = lead times
(+30 min, +3 h, +6 h). Shared colour scale per variable.

In [ ]:
# ── Select models and leads ──────────────────────────────────────────────────
MAP_RUNS = ["v27", "v30-nll"]
MAP_RUNS = [r for r in MAP_RUNS if r in MR0_RUNS]
SHOW_LEADS = [(1, "+30 min"), (6, "+3 h"), (12, "+6 h")]
N_MOD = len(MAP_RUNS); N_LEAD = len(SHOW_LEADS)

for vi, v in enumerate(VARS):
    all_mae = []
    for r in MAP_RUNS:
        a = AGG[r]
        cnt = a["mod_all_cnt"][:, :, vi]
        s   = a["mod_all_sum_phys"][:, :, vi]
        all_mae.append(np.where(cnt > 0, s / np.maximum(cnt, 1), np.nan))

    fig, axes = plt.subplots(N_MOD, N_LEAD,
                             figsize=(5.4 * N_LEAD, 4.2 * N_MOD))
    if N_MOD == 1: axes = axes[None, :]

    vmin = np.nanpercentile(np.concatenate([m.ravel() for m in all_mae]), 2)
    vmax = np.nanpercentile(np.concatenate([m.ravel() for m in all_mae]), 98)
    sc_last = None

    for ri, r in enumerate(MAP_RUNS):
        mae = all_mae[ri]
        label, col, _ = C.MODELS[r]
        for ci, (ki, lead_lbl) in enumerate(SHOW_LEADS):
            ax = axes[ri, ci]
            draw_dem(ax, dem_bg, dem_fg, switzerland)
            vals = mae[ki]
            valid = ~np.isnan(vals)
            sc_last = ax.scatter(
                stn.easting[valid], stn.northing[valid],
                c=vals[valid], s=50, cmap="YlOrRd",
                vmin=vmin, vmax=vmax,
                edgecolors="white", linewidths=0.4, zorder=5)
            overlay_exclusion_markers(ax, v, legend=(ri == 0 and ci == 0))
            if ri == 0:
                ax.set_title(lead_lbl, fontsize=12, fontweight="bold")
            if ci == 0:
                ax.text(-0.06, 0.5, label, transform=ax.transAxes,
                        rotation=90, va="center", ha="center",
                        fontsize=10, fontweight="bold")

    cbar = fig.colorbar(sc_last, ax=axes.ravel().tolist(),
                        location="right", fraction=0.015, pad=0.03,
                        label=f"MAE [{C.UNITS[v]}]", shrink=0.7,
                        aspect=30)
    fig.suptitle(f"{v} — per-station MAE at MR=0.0", fontsize=14, y=1.02)

    # ── Sync axes limits across all subplots ──
    for ci in range(N_LEAD):
        col_axes = [axes[ri, ci] for ri in range(N_MOD)]
        lo_y = min(ax.get_ylim()[0] for ax in col_axes)
        hi_y = max(ax.get_ylim()[1] for ax in col_axes)
        lo_x = min(ax.get_xlim()[0] for ax in col_axes)
        hi_x = max(ax.get_xlim()[1] for ax in col_axes)
        for ax in col_axes:
            ax.set_ylim(lo_y, hi_y)
            ax.set_xlim(lo_x, hi_x)

    fig.subplots_adjust(right=0.88, wspace=0.05, hspace=0.08)
    C.save_fig(fig, f"45_map_{v}"); plt.show()
    plt.close(fig)

# Free unused AGG entries
for _r in list(AGG):
    if _r not in MAP_RUNS:
        del AGG[_r]
import gc; gc.collect()


In [ ]:
# ── Per-station σₑ maps — same layout as MAE maps ───────────────────────────
for vi, v in enumerate(VARS):
    all_sd = []
    for r in MAP_RUNS:
        a = AGG[r]
        cnt = a["mod_all_cnt"][:, :, vi]
        sgn = a["mod_all_signed_phys"][:, :, vi]
        sq  = a["mod_all_sumsq_phys"][:, :, vi]
        n = np.maximum(cnt, 1)
        sd = np.sqrt(np.maximum(sq / n - (sgn / n) ** 2, 0))
        all_sd.append(np.where(cnt > 0, sd, np.nan))

    fig, axes = plt.subplots(N_MOD, N_LEAD,
                             figsize=(5.4 * N_LEAD, 4.2 * N_MOD))
    if N_MOD == 1: axes = axes[None, :]

    vmin = np.nanpercentile(np.concatenate([s.ravel() for s in all_sd]), 2)
    vmax = np.nanpercentile(np.concatenate([s.ravel() for s in all_sd]), 98)
    sc_last = None

    for ri, r in enumerate(MAP_RUNS):
        sd_vals = all_sd[ri]
        label, col, _ = C.MODELS[r]
        for ci, (ki, lead_lbl) in enumerate(SHOW_LEADS):
            ax = axes[ri, ci]
            draw_dem(ax, dem_bg, dem_fg, switzerland)
            vals = sd_vals[ki]
            valid = ~np.isnan(vals)
            sc_last = ax.scatter(
                stn.easting[valid], stn.northing[valid],
                c=vals[valid], s=50, cmap="magma",
                vmin=vmin, vmax=vmax,
                edgecolors="white", linewidths=0.4, zorder=5)
            if ri == 0:
                ax.set_title(lead_lbl, fontsize=12, fontweight="bold")
            if ci == 0:
                ax.text(-0.06, 0.5, label, transform=ax.transAxes,
                        rotation=90, va="center", ha="center",
                        fontsize=10, fontweight="bold")

    cbar = fig.colorbar(sc_last, ax=axes.ravel().tolist(),
                        location="right", fraction=0.015, pad=0.03,
                        label=f"σₑ [{C.UNITS[v]}]", shrink=0.7,
                        aspect=30)
    fig.suptitle(f"{v} — per-station σₑ at MR=0.0", fontsize=14, y=1.02)

    # ── Sync axes limits across all subplots ──
    for ci in range(N_LEAD):
        col_axes = [axes[ri, ci] for ri in range(N_MOD)]
        lo_y = min(ax.get_ylim()[0] for ax in col_axes)
        hi_y = max(ax.get_ylim()[1] for ax in col_axes)
        lo_x = min(ax.get_xlim()[0] for ax in col_axes)
        hi_x = max(ax.get_xlim()[1] for ax in col_axes)
        for ax in col_axes:
            ax.set_ylim(lo_y, hi_y)
            ax.set_xlim(lo_x, hi_x)

    fig.subplots_adjust(right=0.88, wspace=0.05, hspace=0.08)
    C.save_fig(fig, f"45_sd_map_{v}"); plt.show()
    plt.close(fig)


## Error standard deviation (σₑ) maps — v27 at key leads

In [ ]:
a = AGG["v27"]
cnt = a["mod_all_cnt"]           # (K, N, V)
sgn = a["mod_all_signed_phys"]
rmse_sq = a["mod_all_sumsq_phys"]
n = np.maximum(cnt, 1)
sd = np.sqrt(np.maximum(rmse_sq / n - (sgn / n) ** 2, 0))  # (K, N, V)

for ki, klbl in [(1, "+30 min"), (4, "+2 h"), (12, "+6 h")]:
    fig, axes = plt.subplots(1, NV, figsize=(5.4 * NV, 4.5))
    for vi, (ax, v) in enumerate(zip(axes, VARS)):
        draw_dem(ax, dem_bg, dem_fg, switzerland)
        vals = sd[ki, :, vi]
        valid = cnt[ki, :, vi] > 0
        sc = ax.scatter(stn.easting[valid], stn.northing[valid],
                        c=vals[valid], s=50, cmap="magma",
                        edgecolors="white", linewidths=0.4, zorder=5)
        overlay_exclusion_markers(ax, v, legend=(vi == 0))
        fig.colorbar(sc, ax=ax, fraction=0.04, pad=0.02,
                     label=f"σₑ [{C.UNITS[v]}]")
        ax.set_title(f"{v}", fontsize=10)

    # ── Sync axes limits across all variable subplots ──
    lo_y = min(ax.get_ylim()[0] for ax in axes)
    hi_y = max(ax.get_ylim()[1] for ax in axes)
    lo_x = min(ax.get_xlim()[0] for ax in axes)
    hi_x = max(ax.get_xlim()[1] for ax in axes)
    for ax in axes:
        ax.set_ylim(lo_y, hi_y)
        ax.set_xlim(lo_x, hi_x)

    # ── Sync wind component color scales ──
    wu_i, wv_i = VARS.index("wind_u"), VARS.index("wind_v")
    sc_u = axes[wu_i].collections[-1]
    sc_v = axes[wv_i].collections[-1]
    vmin = min(sc_u.get_clim()[0], sc_v.get_clim()[0])
    vmax = max(sc_u.get_clim()[1], sc_v.get_clim()[1])
    sc_u.set_clim(vmin, vmax)
    sc_v.set_clim(vmin, vmax)

    fig.suptitle(f"MAE Transformer σₑ = std(ŷ − y) at {klbl}  (MR=0.0)", fontsize=13, y=1.02)
    plt.tight_layout(); C.save_fig(fig, f"45_sd_map_{ki}"); plt.show()
    plt.close(fig)


In [ ]:
a = AGG["v30-nll"]
cnt = a["mod_all_cnt"]           # (K, N, V)
sgn = a["mod_all_signed_phys"]
rmse_sq = a["mod_all_sumsq_phys"]
n = np.maximum(cnt, 1)
sd = np.sqrt(np.maximum(rmse_sq / n - (sgn / n) ** 2, 0))  # (K, N, V)

for ki, klbl in [(1, "+30 min"), (4, "+2 h"), (12, "+6 h")]:
    fig, axes = plt.subplots(1, NV, figsize=(5.4 * NV, 4.5))
    for vi, (ax, v) in enumerate(zip(axes, VARS)):
        draw_dem(ax, dem_bg, dem_fg, switzerland)
        vals = sd[ki, :, vi]
        valid = cnt[ki, :, vi] > 0
        sc = ax.scatter(stn.easting[valid], stn.northing[valid],
                        c=vals[valid], s=50, cmap="magma",
                        edgecolors="white", linewidths=0.4, zorder=5)
        overlay_exclusion_markers(ax, v, legend=(vi == 0))
        fig.colorbar(sc, ax=ax, fraction=0.04, pad=0.02,
                     label=f"σₑ [{C.UNITS[v]}]")
        ax.set_title(f"{v}", fontsize=10)

    # ── Sync axes limits across all variable subplots ──
    lo_y = min(ax.get_ylim()[0] for ax in axes)
    hi_y = max(ax.get_ylim()[1] for ax in axes)
    lo_x = min(ax.get_xlim()[0] for ax in axes)
    hi_x = max(ax.get_xlim()[1] for ax in axes)
    for ax in axes:
        ax.set_ylim(lo_y, hi_y)
        ax.set_xlim(lo_x, hi_x)

    # ── Sync wind component color scales ──
    wu_i, wv_i = VARS.index("wind_u"), VARS.index("wind_v")
    sc_u = axes[wu_i].collections[-1]
    sc_v = axes[wv_i].collections[-1]
    vmin = min(sc_u.get_clim()[0], sc_v.get_clim()[0])
    vmax = max(sc_u.get_clim()[1], sc_v.get_clim()[1])
    sc_u.set_clim(vmin, vmax)
    sc_v.set_clim(vmin, vmax)

    fig.suptitle(f"Probabilistic MAE σₑ = std(ŷ − y) at {klbl}  (MR=0.0)", fontsize=13, y=1.02)
    plt.tight_layout(); C.save_fig(fig, f"45_sd_map_{ki}"); plt.show()
    plt.close(fig)


In [ ]:
# ── v27 vs v30-nll  σₑ comparison — temperature & pressure ──────────────────
# Rows = models (MAE, Probabilistic MAE), columns = lead times (+30 min, +3 h, +6 h)
# One figure per variable (temperature, pressure only)

CMP_RUNS  = ["v27", "v30-nll"]
CMP_LEADS = [(1, "+30 min"), (6, "+3 h"), (12, "+6 h")]
CMP_VARS  = ["temperature", "pressure"]
N_CMP     = len(CMP_RUNS)
N_CLEAD   = len(CMP_LEADS)

for v in CMP_VARS:
    vi = VARS.index(v)
    all_sd = []
    for r in CMP_RUNS:
        a   = AGG[r]
        cnt = a["mod_all_cnt"][:, :, vi]
        sgn = a["mod_all_signed_phys"][:, :, vi]
        sq  = a["mod_all_sumsq_phys"][:, :, vi]
        n   = np.maximum(cnt, 1)
        sd  = np.sqrt(np.maximum(sq / n - (sgn / n) ** 2, 0))
        all_sd.append(np.where(cnt > 0, sd, np.nan))

    # Shared colour scale across both models and all lead times.
    # `list(zip(*CMP_LEADS))[0]` is a Python TUPLE of lead indices, e.g.
    # (1, 6, 12) — numpy treats a tuple index as multi-axis advanced
    # indexing (sd[1, 6, 12]), not "select these rows along axis 0", so
    # this raised "IndexError: too many indices for array" on a 2-D sd.
    # A list index does the intended row selection.
    CMP_LEAD_IDX = [ki for ki, _ in CMP_LEADS]
    vmin = np.nanpercentile(np.concatenate(
        [sd[CMP_LEAD_IDX].ravel() for sd in all_sd]), 2)
    vmax = np.nanpercentile(np.concatenate(
        [sd[CMP_LEAD_IDX].ravel() for sd in all_sd]), 98)

    fig, axes = plt.subplots(N_CMP, N_CLEAD,
                             figsize=(5.4 * N_CLEAD, 4.2 * N_CMP))
    if N_CMP == 1: axes = axes[None, :]
    sc_last = None

    for ri, r in enumerate(CMP_RUNS):
        sd_vals = all_sd[ri]
        label, col, _ = C.MODELS[r]
        for ci, (ki, lead_lbl) in enumerate(CMP_LEADS):
            ax = axes[ri, ci]
            draw_dem(ax, dem_bg, dem_fg, switzerland)
            vals = sd_vals[ki]
            valid = ~np.isnan(vals)
            sc_last = ax.scatter(
                stn.easting[valid], stn.northing[valid],
                c=vals[valid], s=50, cmap="magma",
                vmin=vmin, vmax=vmax,
                edgecolors="white", linewidths=0.4, zorder=5)
            if ri == 0:
                ax.set_title(lead_lbl, fontsize=12, fontweight="bold")
            if ci == 0:
                ax.text(-0.06, 0.5, label, transform=ax.transAxes,
                        rotation=90, va="center", ha="center",
                        fontsize=10, fontweight="bold")

    # ── Sync axes limits ──
    for ci in range(N_CLEAD):
        col_axes = [axes[ri, ci] for ri in range(N_CMP)]
        lo_y = min(a.get_ylim()[0] for a in col_axes)
        hi_y = max(a.get_ylim()[1] for a in col_axes)
        lo_x = min(a.get_xlim()[0] for a in col_axes)
        hi_x = max(a.get_xlim()[1] for a in col_axes)
        for a in col_axes:
            a.set_ylim(lo_y, hi_y)
            a.set_xlim(lo_x, hi_x)

    cbar = fig.colorbar(sc_last, ax=axes.ravel().tolist(),
                        location="right", fraction=0.015, pad=0.03,
                        label=f"σₑ [{C.UNITS[v]}]", shrink=0.7,
                        aspect=30)
    fig.suptitle(f"{v}: per-station error SD σₑ = std(ŷ−y) at three lead times, all stations visible (MR=0) — rows: MAE Transformer (Huber), Probabilistic MAE (Gaussian NLL)",
                 fontsize=14, y=1.02)
    fig.subplots_adjust(right=0.88, wspace=0.05, hspace=0.08)
    C.save_fig(fig, f"45_sd_cmp_{v}"); plt.show()
    plt.close(fig)


## Interpretation

Stations with persistently high MAE across models tend to be isolated
or in complex terrain. The DEM background reveals whether high-error
stations cluster in Alpine valleys or on exposed ridges. σₑ
maps highlight stations where the model is occasionally very wrong
(high variability), even if the mean error is moderate.

## Stations excluded from evaluation, by variable

Marks every station×variable pair dropped by `DROP_SV` (see common.py) with
an X on the map: **black** = missing in both train and test (>50%),
**white** = missing in train only (sensor added after 2021, present in
test), **grey** = missing in test only (no such pair exists in the current
data — kept for completeness), **orange** = present in both splits but
excluded for a data-quality reason (BIZ/pressure, systematic sensor drift),
not for missingness.

In [ ]:
# ── Stations excluded from evaluation, by variable ──────────────────────────
REASON_STYLE = {
    "missing_train": dict(color="white",   label="missing in train only"),
    "missing_test":  dict(color="0.6",     label="missing in test only"),
    "missing_both":  dict(color="black",   label="missing in train & test"),
    "drift":         dict(color="#E8A838", label="excluded \u2014 sensor drift"),
}
EXCL_REASONS = C.excluded_station_variable_reasons()

fig, axes = plt.subplots(1, NV, figsize=(6.0 * NV, 7.5))
for vi, (ax, v) in enumerate(zip(axes, VARS)):
    draw_dem(ax, dem_bg, dem_fg, switzerland)
    ax.set_title(f"{v}", fontsize=12)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=len(REASON_STYLE),
          fontsize=9, bbox_to_anchor=(0.5, -0.02), frameon=True)
fig.suptitle("Stations excluded from evaluation, by variable", y=1.02)
plt.tight_layout()
C.save_fig(fig, "excluded_stations_map")
plt.show()
plt.close(fig)

print("Excluded station\u00d7variable pairs:")
for (abbr, v), reason in sorted(EXCL_REASONS.items()):
    print(f"  {abbr:<4} {v:<12} {reason}")